# Confirm signal acquisition of viable satellites

- Load the *viable* satellites from step 3
- Correlate preprocessed voltage streams with relevant GPS Gold Code PRN filters
- Search over doppler to find the correct carrier doppler
- Save the results into an acquisition table

In [1]:
import sys
import os
import json
# Get parent directory
parent_dir = os.path.abspath("..")
# Add it to sys.path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from pipeline.GPS import GPS_handler
from pipeline.dataloader import raw_data_loader
from pipeline.JSON_handler import load_from_json, to_json_safe, save_to_json
from pipeline.acquisition_handler import acquire_GPS 

In [2]:
TARTS = {
    'rhodes': 'za-rhodes',
    'namibia': 'na-unam',
}

DATASETS = {
    'rhodes': 'rhodes/data_2026-02-13_14_39_25.231047.hdf',
    'namibia': 'namibia/data_2026-02-13_14_39_25.193869.hdf'
}

t_obs = 1770993565.3405828

filepath = f"/home/jdawson/repos/TART/notebooks/dataproducts/{t_obs}"

## Load pre-processed voltage streams from Rhodes and Namibia

In [3]:
loader = raw_data_loader("/home/jdawson/repos/TART/datasets")
data_rhodes, time_rhodes, Fs_rhodes = loader.load_preprocessed_data(DATASETS['rhodes'])
data_namibia, time_namibia, Fs_namibia = loader.load_preprocessed_data(DATASETS['namibia'])
Fs_baseband = data_rhodes['Fs']

# Get the data ready in the form of a dictionary for the acquisition handler:

TART_DATA = {
    'rhodes': {
        'data': data_rhodes['data'],
        'Fs': data_rhodes['Fs'],
    },
    'namibia': {
        'data': data_namibia['data'],
        'Fs': data_namibia['Fs'],
    }
}

Loading: rhodes/data_2026-02-13_14_39_25.231047.hdf...
Completed.
Loading: namibia/data_2026-02-13_14_39_25.193869.hdf...
Completed.


## Load the viable satellites for these two sites

In [4]:
filename = 'GPS_viable_satellites.json'
viable_satellites = load_from_json(filepath, filename)
print(viable_satellites)

{'namibia': ['G10', 'G23', 'G25', 'G26', 'G28', 'G31', 'G32'], 'rhodes': ['G10', 'G12', 'G23', 'G25', 'G28', 'G31', 'G32'], 'mauritius': ['G10', 'G12', 'G15', 'G23', 'G24', 'G25', 'G28', 'G29', 'G32']}


In [5]:
handler = GPS_handler()
viable_satellites_union = handler.get_union_satellites(viable_satellites, TARTS)
print('Union of viable satellites for current TARTS are:', viable_satellites_union)

Union of viable satellites for current TARTS are: ['G10', 'G23', 'G25', 'G28', 'G31', 'G32']


## Set up appropriately sampled GPS gold-code PRNs for correlation

In [6]:
# Instantiate the acquisition handler for this stage
acquisition_handler = acquire_GPS(Fs=Fs_baseband)

In [7]:
# Generate PRN filters for all 32 GPS satellites
prn_codes = acquisition_handler.generate_all_prn_codes()
# prn_codes is a dictionary of key:value pairs of the form ID:[data], where ID is the numerical value of GPS IDs
# i.e.: 1, 2, 3... 32.

Pre-computing PRN codes for all satellites...
Done.


## Run the carrier doppler acquisition search 

In [8]:
# Run the carrier doppler acquisition using either single or multithreaded approach.
# Uncomment to choose:

# acquisition_results = acquisition_handler.acquire_satellites(TART_DATA, viable_satellites_union, prn_codes)
acquisition_results_pooled = acquisition_handler.acquire_satellites_pool(TART_DATA, viable_satellites_union, prn_codes)

Running parallel acquisition for 6 satellites (12 jobs, auto workers)...


Acquiring satellites: 100%|█████████████████████████████████████████████████████████████████████| 12/12 [01:22<00:00,  6.87s/it]

Acquisition complete!


Take a look at the acquisitionm results dictionary below. 

For each TART and each viable satellite, the maximum SNR for each antenna doppler search as well as the doppler at that position are recorded.

It's clear where a true acquisition has been made as the SNRs are >>5 and the doppler values mostly agree.

In [9]:
acquisition_results_pooled

{'rhodes': {'G10': {'detected': False,
   'doppler': np.int64(-3250),
   'raw_detected_doppler': array([-4250.,  4750., -4000., -4250.,     0.,  4250.,   750.,  3000.,
           2500., -3000., -1750.,  -500., -2000.,  1000., -3000.,  2250.,
            250., -1500., -2500., -3250., -3250.,  -500., -3250., -1500.]),
   'snrs': array([2.21573806, 2.17854023, 2.09233236, 1.97411335, 2.10763669,
          2.16478968, 2.15568638, 1.89220047, 2.15205073, 2.50874591,
          2.0054028 , 2.27897763, 2.16573858, 2.17986345, 1.72869813,
          2.5329268 , 2.21433568, 2.05494571, 2.21509409, 2.36998367,
          2.23434258, 2.00255203, 3.23480439, 2.26796484]),
   'best_antenna': 22},
  'G23': {'detected': True,
   'doppler': np.int64(3000),
   'raw_detected_doppler': array([ 3000.,  3000.,  3000.,  3000.,  3000., -2750.,  3000.,  3000.,
           3000.,  3000.,  3000.,  3000.,  3000.,  3000.,  4250.,  3000.,
           3000.,  3000.,  2250.,  3000.,  3000.,  3250.,  3000.,  3000.]),
   '

## Save the results to disk

In [10]:
filename = f"GPS_acquisition_results.json"
serialisable = to_json_safe(acquisition_results_pooled)
save_to_json(serialisable, filepath, filename)

'JSON saved to disk.'